# View your scheduler trace

Optional companion to stage 07. Export your own completed `engine.history` as `artifacts/learner-trace.json` using `json.dumps`. This notebook reads that artifact; it does not run the instructor engine. Missing trace data is reported explicitly.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
path = root / "artifacts/learner-trace.json"
if not path.exists():
    print("No learner trace yet. Build stage 07 and export engine.history first.")
else:
    history = json.loads(path.read_text())
    ids = sorted({row["request_id"] for step in history for row in step["scheduled"]})
    fig, ax = plt.subplots(figsize=(10, max(3, len(ids)*0.6)))
    for i, step in enumerate(history):
        for row in step["scheduled"]:
            y = ids.index(row["request_id"])
            color = "#2563eb" if row["phase"] == "prefill" else "#16a34a"
            ax.barh(y, .85, left=i, color=color)
            ax.text(i+.42, y, str(row["count"]), ha="center", va="center", color="white")
    ax.set_yticks(range(len(ids)), ids)
    ax.set_xlabel("Execution step (not elapsed time)")
    ax.set_title("Your schedule: blue prefill, green decode, labels are token counts")
    plt.tight_layout()
    plt.show()

Look for late admissions, prompt chunks without emissions, and capacity freed by completion. A step index is not a hardware utilization or latency measurement. Explain each row using your request state transitions.